In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "project_paths.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "project_paths.py").exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)


In [1]:
# Install packages (run once)
!pip install transformers datasets torch accelerate scikit-learn pandas openpyxl -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

d:\PGDBA\MyNotes\SEM-2\DSLab\DSL_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load data - FIXED PATH
df = pd.read_excel("data/raw/Book1.xlsx")
print("Columns:", df.columns.tolist())
print(df.head())
print("Severity distribution:")
print(df['severity'].value_counts())

Columns: ['severity', 'grievance_text']
   severity                            grievance_text
0      High               please still keeps dropping
1  Critical     please facing no internet access yaar
2       Low           please still router not working
3    Medium               facing data loss issue yaar
4  Very Low  INTERNET VERY SLOW SINCE MORNING!!! yaar
Severity distribution:
severity
High        15874
Medium      14409
Critical     7369
Very Low     6215
Low          6133
Name: count, dtype: int64


In [4]:
# Split data
X = df.drop(columns=['severity'])
y = df['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (40000, 1), Test shape: (10000, 1)


In [5]:
# Label encoding
unique_labels = sorted(y_train.unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", unique_labels)

Labels: ['Critical', 'High', 'Low', 'Medium', 'Very Low']


In [6]:
# Train dataset
y_train_encoded = y_train.map(label2id)
train_df = pd.DataFrame({
    "text": X_train['grievance_text'],
    "labels": y_train_encoded
})

train_dataset = Dataset.from_pandas(train_df)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

d:\PGDBA\MyNotes\SEM-2\DSLab\DSL_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\karth\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 40000/40000 [00:02<00:00, 17431.14 examples/s]


ValueError: Original column name label not in the dataset. Current columns in the dataset: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']

In [ ]:
# Model and training
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
# Test dataset and evaluation - FIXED
y_test_encoded = y_test.map(label2id)
test_df = pd.DataFrame({
    "text": X_test['grievance_text'],
    "labels": y_test_encoded
})

test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Evaluate - FIXED
results = trainer.evaluate(test_dataset)
print("Evaluation results:", results)

# Metrics
predictions_output = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

accuracy = accuracy_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print("Original Untitled20 fixed!")